# Product Visual Search: full Colab pipeline

This notebook runs the complete project pipeline for submission:

- ResNet18 supervised training and retrieval evaluation
- Frozen CLIP retrieval evaluation
- Lightly fine-tuned CLIP image encoder using `articleType` supervision

Expected Google Drive files:

- `MyDrive/product_visual_search/project.zip`
- `MyDrive/product_visual_search/fashion-product-images-small.zip`


## 1. Mount Google Drive

Before running the notebook, set Colab runtime to GPU.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Configuration

In [2]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/product_visual_search_v2')
PROJECT_ZIP = DRIVE_ROOT / 'project.zip'
DATA_ZIP = DRIVE_ROOT / 'fashion-product-images-small.zip'
WORKDIR = Path('/content/product_visual_search')

# Full submission run settings.
EPOCHS = 20
RESUME_TRAINING = False

# Use these switches if you need to rerun only one part.
RUN_RESNET = True
RUN_FROZEN_CLIP = True
RUN_CLIP_FINETUNE = True

RESNET_BATCH_SIZE = 64
CLIP_BATCH_SIZE = 16
NUM_WORKERS = 2
UNFREEZE_VISUAL_BLOCKS = 1

print('PROJECT_ZIP:', PROJECT_ZIP)
print('DATA_ZIP:', DATA_ZIP)
print('WORKDIR:', WORKDIR)

PROJECT_ZIP: /content/drive/MyDrive/product_visual_search_v2/project.zip
DATA_ZIP: /content/drive/MyDrive/product_visual_search_v2/fashion-product-images-small.zip
WORKDIR: /content/product_visual_search


## 3. Check GPU and uploaded files

In [3]:
import torch

print('cuda_available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'Please switch Colab runtime to GPU before full training.'
assert PROJECT_ZIP.exists(), f'Missing {PROJECT_ZIP}'
assert DATA_ZIP.exists(), f'Missing {DATA_ZIP}'
print('project.zip MB:', round(PROJECT_ZIP.stat().st_size / 1024 / 1024, 2))
print('dataset.zip MB:', round(DATA_ZIP.stat().st_size / 1024 / 1024, 2))

cuda_available: True
device: Tesla T4
project.zip MB: 2.63
dataset.zip MB: 565.16


## 4. Unpack project and dataset

In [4]:
import shutil
import zipfile

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
WORKDIR.mkdir(parents=True, exist_ok=True)

tmp_dir = Path('/content/project_unpack')
if tmp_dir.exists():
    shutil.rmtree(tmp_dir)
tmp_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(PROJECT_ZIP, 'r') as zf:
    zf.extractall(tmp_dir)

children = list(tmp_dir.iterdir())
source_root = children[0] if len(children) == 1 and children[0].is_dir() else tmp_dir
for item in source_root.iterdir():
    target = WORKDIR / item.name
    if target.exists():
        if target.is_dir():
            shutil.rmtree(target)
        else:
            target.unlink()
    shutil.move(str(item), str(target))

raw_dir = WORKDIR / 'data' / 'raw'
raw_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(DATA_ZIP, 'r') as zf:
    zf.extractall(raw_dir)

myntra_dir = raw_dir / 'myntradataset'
if (myntra_dir / 'styles.csv').exists():
    shutil.copy2(myntra_dir / 'styles.csv', raw_dir / 'styles.csv')
if (myntra_dir / 'images').exists():
    target_images = raw_dir / 'images'
    if target_images.exists():
        shutil.rmtree(target_images)
    shutil.copytree(myntra_dir / 'images', target_images)

print('project root:', sorted(p.name for p in WORKDIR.iterdir()))
print('prepare split exists:', (WORKDIR / 'scripts' / 'prepare_full_dataset_split.py').exists())
print('styles.csv exists:', (raw_dir / 'styles.csv').exists())
print('image_count:', len(list((raw_dir / 'images').glob('*.jpg'))))

project root: ['.gitattributes', '.gitignore', 'GITHUB_UPLOAD_INSTRUCTIONS.md', 'MODEL_AND_DATA_FILES.md', 'README.md', 'app.py', 'colab_all_models_full_pipeline.ipynb', 'colab_clip_articletype_clean.ipynb', 'colab_clip_articletype_pipeline.ipynb', 'data', 'generate_report_docx.py', 'outputs', 'product_visual_search_retrieval.ipynb', 'requirements.txt', 'scripts', 'src']
prepare split exists: True
styles.csv exists: True
image_count: 44441


## 5. Install dependencies

In [5]:
%cd /content/product_visual_search
!python --version
!pip install -r requirements.txt

/content/product_visual_search
Python 3.12.13
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 95.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.4 MB/s eta 0:00:00


## 5b. Runtime helpers and package sanity check


In [6]:
import subprocess
import sys


def run_cmd(command):
    if command.startswith('python '):
        command = command.replace('python ', 'python -u ', 1)
    print(command, flush=True)
    process = subprocess.Popen(
        command,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'Command failed with exit code {return_code}: {command}')


## 6. Prepare split

In [7]:
%cd /content/product_visual_search
run_cmd('python scripts/prepare_full_dataset_split.py --skip-image-verify')


/content/product_visual_search
python -u scripts/prepare_full_dataset_split.py --skip-image-verify
valid_image_count=44405
num_classes=132
train_size=31070
val_size=6666
test_size=6669
dropped_rare_classes_count=10
missing_image_metadata_rows=5
unreadable_images=0
images_per_class_top20=
          articleType  image_count
              Tshirts         7066
               Shirts         3215
         Casual Shoes         2845
              Watches         2542
         Sports Shoes         2036
               Kurtas         1844
                 Tops         1762
             Handbags         1759
                Heels         1323
           Sunglasses         1073
              Wallets          936
           Flip Flops          914
              Sandals          897
               Briefs          849
                Belts          813
            Backpacks          724
                Socks          686
         Formal Shoes          637
Perfume and Body Mist          613
           

## 7. Ensure local CLIP checkpoint

The frozen CLIP scripts expect the Hugging Face safetensors file at the project-local model path.

In [8]:
from huggingface_hub import hf_hub_download

local_clip = WORKDIR / 'models' / 'huggingface' / 'timm_vit_base_patch32_clip_224_openai' / 'open_clip_model.safetensors'
local_clip.parent.mkdir(parents=True, exist_ok=True)
if not local_clip.exists():
    downloaded = hf_hub_download(
        repo_id='timm/vit_base_patch32_clip_224.openai',
        filename='open_clip_model.safetensors',
    )
    shutil.copy2(downloaded, local_clip)
print('local_clip:', local_clip)
print('exists:', local_clip.exists())
print('size MB:', round(local_clip.stat().st_size / 1024 / 1024, 2))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

local_clip: /content/product_visual_search/models/huggingface/timm_vit_base_patch32_clip_224_openai/open_clip_model.safetensors
exists: True
size MB: 577.11


## 8. Train and evaluate ResNet18

In [9]:
if RUN_RESNET:
    resume = ' --resume' if RESUME_TRAINING else ''
    cmd = (
        f"python scripts/train_resnet18_full.py --epochs {EPOCHS} --batch-size {RESNET_BATCH_SIZE} "
        f"--num-workers {NUM_WORKERS} --patience {EPOCHS}{resume}"
    )
    run_cmd(cmd)
else:
    print('ResNet training skipped')


流式输出内容被截断，只能显示最后 5000 行内容。
resnet train epoch 12/20: 100%|██████████| 486/486 [02:09<00:00,  3.74it/s]

resnet val epoch 12/20: 100%|██████████| 105/105 [00:15<00:00,  6.62it/s]
epoch=12 train_loss=0.0474 val_loss=0.4601 val_top1=0.8888 val_top5=0.9889 epoch_time=145.8s lr=0.0001

resnet train epoch 13/20: 100%|██████████| 486/486 [02:13<00:00,  3.65it/s]

resnet val epoch 13/20: 100%|██████████| 105/105 [00:15<00:00,  6.93it/s]
epoch=13 train_loss=0.0396 val_loss=0.4994 val_top1=0.8917 val_top5=0.9887 epoch_time=148.5s lr=0.0001

resnet train epoch 14/20: 100%|██████████| 486/486 [02:10<00:00,  3.73it/s]

resnet val epoch 14/20: 100%|██████████| 105/105 [00:15<00:00,  6.82it/s]
epoch=14 train_loss=0.0408 val_loss=0.4834 val_top1=0.8900 val_top5=0.9893 epoch_time=145.8s lr=0.0001

resnet train epoch 15/20: 100%|██████████| 486/486 [02:11<00:00,  3.71it/s]

resnet val epoch 15/20: 100%|██████████| 105/105 [00:16<00:00,  6.45it/s]
epoch=15 train_loss=0.0310 val_loss=0.5005 val_top1=0.885

In [10]:
if RUN_RESNET:
    run_cmd(f'python scripts/evaluate_resnet18_full.py --batch-size {RESNET_BATCH_SIZE} --num-workers {NUM_WORKERS}')
    run_cmd(f'python scripts/build_resnet18_full_index.py --batch-size {RESNET_BATCH_SIZE} --num-workers {NUM_WORKERS}')
    run_cmd(f'python scripts/evaluate_resnet18_full_retrieval.py --batch-size {RESNET_BATCH_SIZE} --num-workers {NUM_WORKERS}')
else:
    print('ResNet evaluation skipped')


python -u scripts/evaluate_resnet18_full.py --batch-size 64 --num-workers 2
{
  "test_loss": 0.41876709798158496,
  "test_top1_acc": 0.8947368421052632,
  "test_top5_acc": 0.9880041985305144,
  "test_size": 6669,
  "train_size": 31070,
  "val_size": 6666,
  "num_classes": 132,
  "checkpoint": "/content/product_visual_search/outputs/checkpoints/resnet18_full_best.pth"
}
metrics_json=/content/product_visual_search/outputs/reports/resnet18_full_test_metrics.json
per_class_accuracy_csv=/content/product_visual_search/outputs/reports/resnet18_full_per_class_accuracy.csv
summary_csv_updated=/content/product_visual_search/outputs/reports/experiment_summary.csv
python -u scripts/build_resnet18_full_index.py --batch-size 64 --num-workers 2
gallery_size=31070
embedding_shape=(31070, 512)
index_scope=full train gallery
output_path=/content/product_visual_search/outputs/indexes/resnet18_full_gallery_index.npz
python -u scripts/evaluate_resnet18_full_retrieval.py --batch-size 64 --num-workers 2
{
  

In [11]:
resnet_backup = DRIVE_ROOT / 'outputs_after_resnet'
resnet_backup.mkdir(parents=True, exist_ok=True)
!cp -r /content/product_visual_search/outputs {resnet_backup}
print('Backed up ResNet stage to', resnet_backup)

Backed up ResNet stage to /content/drive/MyDrive/product_visual_search_v2/outputs_after_resnet


## 9. Evaluate frozen CLIP

In [12]:
if RUN_FROZEN_CLIP:
    run_cmd('python scripts/build_clip_full_index.py')
    run_cmd(f'python scripts/evaluate_clip_full_retrieval.py --batch-size {CLIP_BATCH_SIZE}')
else:
    print('Frozen CLIP skipped')


python -u scripts/build_clip_full_index.py
checkpoint_source=/content/product_visual_search/models/huggingface/timm_vit_base_patch32_clip_224_openai/open_clip_model.safetensors
download_new_model=False
index_path=/content/product_visual_search/outputs/indexes/clip_full_gallery_index.npz
gallery_size=31070
embedding_shape=(31070, 512)
query_size=6669
num_classes=132
label_column=articleType
index_scope=full train gallery
python -u scripts/evaluate_clip_full_retrieval.py --batch-size 16
{
  "recall@1": 0.8289098815414605,
  "recall@5": 0.9542660068975859,
  "recall@10": 0.9725596041385515,
  "query_size": 6669,
  "gallery_size": 31070,
  "num_classes": 132,
  "embedding_dim": 512,
  "embedding_extraction_time_seconds": 34.789523642999484,
  "checkpoint_source": "/content/product_visual_search/models/huggingface/timm_vit_base_patch32_clip_224_openai/open_clip_model.safetensors",
  "index_path": "/content/product_visual_search/outputs/indexes/clip_full_gallery_index.npz",
  "is_smoke_test"

## 10. Train and evaluate fine-tuned CLIP

In [ ]:
if RUN_CLIP_FINETUNE:
    resume = ' --resume' if RESUME_TRAINING else ''
    cmd = (
        f"python scripts/train_clip_articletype.py --epochs {EPOCHS} --batch-size {CLIP_BATCH_SIZE} "
        f"--num-workers {NUM_WORKERS} --unfreeze-visual-blocks {UNFREEZE_VISUAL_BLOCKS} "
        f"--patience {EPOCHS}{resume}"
    )
    run_cmd(cmd)
else:
    print('Fine-tuned CLIP training skipped')


python -u scripts/train_clip_articletype.py --epochs 20 --batch-size 16 --num-workers 2 --unfreeze-visual-blocks 1 --patience 20
device=cuda
num_classes=132
train_size=31070
val_size=6666
smoke_test=False
trainable_params=7550340
total_params=151345029
trainable_ratio=0.049888

clip-ft train epoch 1/20: 100%|██████████| 1942/1942 [02:26<00:00, 13.26it/s]

clip-ft val epoch 1/20: 100%|██████████| 417/417 [00:27<00:00, 15.06it/s]
epoch=1 train_loss=0.7936 val_loss=0.6086 val_top1=0.8194 val_top5=0.9775 epoch_time=174.2s lr=0.0005

clip-ft train epoch 2/20: 100%|██████████| 1942/1942 [02:28<00:00, 13.08it/s]

clip-ft val epoch 2/20: 100%|██████████| 417/417 [00:27<00:00, 14.98it/s]
epoch=2 train_loss=0.5578 val_loss=0.5613 val_top1=0.8272 val_top5=0.9799 epoch_time=176.4s lr=0.0005

clip-ft train epoch 3/20: 100%|██████████| 1942/1942 [02:28<00:00, 13.06it/s]

clip-ft val epoch 3/20: 100%|██████████| 417/417 [00:27<00:00, 15.03it/s]
epoch=3 train_loss=0.4892 val_loss=0.5351 val_top1=0.828

In [ ]:
if RUN_CLIP_FINETUNE:
    run_cmd(f'python scripts/evaluate_clip_articletype.py --batch-size {CLIP_BATCH_SIZE} --num-workers {NUM_WORKERS}')
    run_cmd(f'python scripts/build_clip_articletype_index.py --batch-size {CLIP_BATCH_SIZE} --num-workers {NUM_WORKERS}')
    run_cmd(f'python scripts/evaluate_clip_articletype_retrieval.py --batch-size {CLIP_BATCH_SIZE} --num-workers {NUM_WORKERS}')
else:
    print('Fine-tuned CLIP evaluation skipped')


## 11. Show summary table

In [ ]:
import pandas as pd

summary_path = WORKDIR / 'outputs' / 'reports' / 'experiment_summary.csv'
summary = pd.read_csv(summary_path)
summary[summary['experiment_mode'].eq('full')][[
    'model_name', 'top1_acc', 'top5_acc', 'recall@1', 'recall@5', 'recall@10',
    'embedding_extraction_time_seconds', 'embedding_dim'
]]

## 12. Back up final outputs

In [ ]:
final_backup = DRIVE_ROOT / 'outputs_all_models_final'
if final_backup.exists():
    shutil.rmtree(final_backup)
final_backup.mkdir(parents=True, exist_ok=True)
!cp -r /content/product_visual_search/outputs {final_backup}
print('Final outputs saved to', final_backup)

## 13. Key files

In [ ]:
key_files = [
    WORKDIR / 'outputs' / 'reports' / 'experiment_summary.csv',
    WORKDIR / 'outputs' / 'reports' / 'resnet18_full_test_metrics.json',
    WORKDIR / 'outputs' / 'reports' / 'resnet18_full_retrieval_metrics.json',
    WORKDIR / 'outputs' / 'reports' / 'clip_full_retrieval_metrics.json',
    WORKDIR / 'outputs' / 'reports' / 'clip_articletype_test_metrics.json',
    WORKDIR / 'outputs' / 'reports' / 'clip_articletype_retrieval_metrics.json',
    WORKDIR / 'outputs' / 'checkpoints' / 'resnet18_full_best.pth',
    WORKDIR / 'outputs' / 'checkpoints' / 'clip_articletype_best.pth',
]
for path in key_files:
    print(path.name, '=>', path.exists())